### A5.4.10. Build Layout Propagation Pass

**Problem:**

Build a graph pass that propagates memory layout information (e.g., row-major vs column-major) through the graph and inserts transpose nodes only where a layout mismatch occurs.

**Explanation:**

Different operations prefer different memory layouts. A matmul might want row-major input, while a convolution might want channel-first. Layout propagation assigns a layout to each edge in the graph and inserts explicit transpose operations only at points where adjacent nodes disagree on layout.

How to build it:

1. Each node declares its preferred layout (e.g., `"row_major"` or `"col_major"`).
2. Walk the graph in topological order.
3. For each node, check the layout of each input. If the input's layout differs from what this node expects, insert a transpose node between them.
4. The transpose node's output layout matches the consumer's expectation.
5. After the pass, all layout mismatches are resolved by explicit transposes.

**Example:**

A `conv` node outputs `channel_first` layout. The next `dense` node expects `row_major`. The pass inserts a `transpose` node between them.

In [ ]:
class Node:
    def __init__(self, name, op_type, layout, inputs=None):
        self.name = name
        self.op_type = op_type
        self.layout = layout
        self.inputs = inputs or []

    def __repr__(self):
        input_names = [node.name for node in self.inputs]
        return f"{self.name}({self.op_type}, layout={self.layout}, inputs={input_names})"


def propagate_layouts(nodes):
    transpose_count = 0
    inserted_nodes = []

    for node in nodes:
        new_inputs = []
        for input_node in node.inputs:
            if input_node.layout != node.layout:
                transpose_name = f"transpose_{input_node.name}_to_{node.layout}"
                transpose = Node(transpose_name, "transpose", node.layout, [input_node])
                inserted_nodes.append(transpose)
                new_inputs.append(transpose)
                transpose_count += 1
            else:
                new_inputs.append(input_node)
        node.inputs = new_inputs

    nodes.extend(inserted_nodes)
    return transpose_count


x = Node("x", "input", "row_major")
conv = Node("conv", "conv2d", "channel_first", [x])
pool = Node("pool", "maxpool", "channel_first", [conv])
dense = Node("dense", "matmul", "row_major", [pool])

all_nodes = [x, conv, pool, dense]

print("Before layout propagation:")
for node in all_nodes:
    print(f"  {node}")

num_transposes = propagate_layouts(all_nodes)

print(f"\nAfter layout propagation ({num_transposes} transposes inserted):")
for node in all_nodes:
    print(f"  {node}")

**References:**

📘 Chen, T. et al. (2018). *TVM: An Automated End-to-End Optimizing Compiler for Deep Learning* — Layout Transformations

---

[⬅️ Previous: Build Common Subexpression Elimination Pass](./09_build_common_subexpression_elimination_pass.ipynb) | [Next: Build Kernel Dispatch Table ➡️](./11_build_kernel_dispatch_table.ipynb)